In [1]:
%pip install requests pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\marti\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


*Importación de los 233 datos de la API*

In [2]:
import requests
import pandas as pd

url = "http://104.225.223.220:8003/api/usuarios/random"

respuesta = requests.get(
    url,
    params={"cantidad": 233}
)

print(respuesta.status_code)

datos = respuesta.json()

df = pd.DataFrame(datos)

print(df.shape)
print(df.head())
print(df.dtypes)

200
(233, 2)
   cantidad                                               data
0       233  {'usuario_id': 915, 'nombre': 'Nancy', 'apelli...
1       233  {'usuario_id': 2097, 'nombre': 'Linda', 'apell...
2       233  {'usuario_id': 2877, 'nombre': None, 'apellido...
3       233  {'usuario_id': 1746, 'nombre': 'Filomena', 'ap...
4       233  {'usuario_id': 2130, 'nombre': 'Abigail', 'ape...
cantidad     int64
data        object
dtype: object


Mostrar información de la columna comprimida data

In [5]:
data_expandida = pd.json_normalize(df["data"])

df_full = pd.concat(
    [df.drop(columns=["data"]).reset_index(drop=True), data_expandida.reset_index(drop=True)],
    axis=1
)

print(df_full.shape)
print(df_full.columns.tolist())
display(df_full.head())

(233, 12)
['cantidad', 'usuario_id', 'nombre', 'apellido', 'email', 'telefono', 'fecha_registro', 'pais', 'ciudad', 'edad', 'genero', 'estado_cuenta']


,cantidad,usuario_id,nombre,apellido,email,telefono,fecha_registro,pais,ciudad,edad,genero,estado_cuenta
0,233,915,Nancy,Noguera,uclavero@example.com,472-130-7423x591,2023-11-26,Chile,Valparaíso,42.0,Otro,Suspendido
1,233,2097,Linda,Bernal,rrico@example.com,+34824504340,2023-05-12,España,Sevilla,18.0,Otro,Activo
2,233,2877,NaN,Becerra,albano75@example.com,(621)634-8443x0534,2026-02-24,Perú,Arequipa,32.0,M,Activo
3,233,1746,Filomena,Leal,NaN,+34 902 663 903,2025-10-03,Chile,NaN,27.0,M,Suspendido
4,233,2130,Abigail,Lluch,tborrego@example.org,+77(2)7204509806,2025-06-05,México,NaN,18.0,Otro,Suspendido


In [6]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

display(df_full)

,cantidad,usuario_id,nombre,apellido,email,telefono,fecha_registro,pais,ciudad,edad,genero,estado_cuenta
0,233,915,Nancy,Noguera,uclavero@example.com,472-130-7423x591,2023-11-26,Chile,Valparaíso,42.0,Otro,Suspendido
1,233,2097,Linda,Bernal,rrico@example.com,+34824504340,2023-05-12,España,Sevilla,18.0,Otro,Activo
2,233,2877,NaN,Becerra,albano75@example.com,(621)634-8443x0534,2026-02-24,Perú,Arequipa,32.0,M,Activo
3,233,1746,Filomena,Leal,NaN,+34 902 663 903,2025-10-03,Chile,NaN,27.0,M,Suspendido
4,233,2130,Abigail,Lluch,tborrego@example.org,+77(2)7204509806,2025-06-05,México,NaN,18.0,Otro,Suspendido
...,...,...,...,...,...,...,...,...,...,...,...,...
228,233,2656,Espartaco,Montero,zoe80@example.net,+34981 75 75 75,2026-03-13,Perú,NaN,25.0,NaN,Activo
229,233,89,Olivia,Martínez,inoriega@example.com,733.484.3443x7575,2024-03-21,España,Bilbao,31.0,F,Activo
230,233,1025,Bartolomé,Polanco,NaN,+34 985 271 012,2026-04-23,Venezuela,Barcelona,NaN,NaN,Activo
231,233,2364,Yuridia,Bernal,marreromario@example.net,NaN,2023-12-14,Chile,Temuco,NaN,F,Activo


Limpieza inicial de los datos

In [ ]:
df_clean = df_full.copy()

df_clean = df_clean.drop_duplicates()

# 2) Convertir posibles numéricas (ajusta según tus columnas reales)
for col in df_clean.columns:
    if df_clean[col].dtype == "object":
        df_clean[col] = pd.to_numeric(df_clean[col], errors="ignore")

print("Filas:", len(df_clean))
print("\nNulos por columna:")
print(df_clean.isna().sum().sort_values(ascending=False).head(15))
print("\nTipos:")
print(df_clean.dtypes)

Filas: 233

Nulos por columna:
ciudad            59
telefono          59
genero            58
nombre            57
email             57
apellido          51
edad              49
usuario_id         0
cantidad           0
fecha_registro     0
pais               0
estado_cuenta      0
dtype: int64

Tipos:
cantidad            int64
usuario_id          int64
nombre                str
apellido              str
email                 str
telefono              str
fecha_registro        str
pais                  str
ciudad                str
edad              float64
genero                str
estado_cuenta         str
dtype: object


Como la mayoria de columnas son categoricas(Texto) trabajaremos con la edad

In [9]:
# Variable(s) numérica(s) válida(s) para outliers (excluye ID y constantes)
cols_outliers = []
for c in df_clean.select_dtypes(include=[np.number]).columns:
    if c not in ["usuario_id"] and df_clean[c].nunique(dropna=True) > 1:
        cols_outliers.append(c)

print("Columnas para outliers:", cols_outliers)

Columnas para outliers: ['edad']


informacion sobre los datos de la columna edad más expandida

In [12]:
print("Total filas:", len(df_clean))
print("Nulos en edad:", df_clean["edad"].isna().sum())
print("Edad no nula:", df_clean["edad"].notna().sum())

Total filas: 233
Nulos en edad: 49
Edad no nula: 184


Se eliminan los datos nulos de ls columna edad

In [ ]:
edad = df_clean["edad"].dropna()

print("N:", len(edad))
print("Media:", round(edad.mean(), 2))
print("Mediana:", round(edad.median(), 2))
print("Moda:", edad.mode().tolist())
print("Desv. estándar:", round(edad.std(ddof=0), 2))
print("Mínimo:", edad.min(), "Máximo:", edad.max())

N: 184
Media: 46.47
Mediana: 48.0
Moda: [48.0, 54.0, 56.0, 71.0]
Desv. estándar: 16.43
Mínimo: 18.0 Máximo: 75.0


N = 184 porque se restaron los datos nulos, es decir, 233 - 49 = 184, por lo tanto se trabaja con esa cantidad de datos

IQR

In [15]:
q1 = edad.quantile(0.25)
q3 = edad.quantile(0.75)
iqr = q3 - q1
li = q1 - 1.5 * iqr
ls = q3 + 1.5 * iqr

out_iqr = edad[(edad < li) | (edad > ls)]

print("Q1:", q1, "Q3:", q3, "IQR:", iqr)
print("Límite inferior:", li, "Límite superior:", ls)
print("Outliers IQR:", len(out_iqr))

Q1: 31.0 Q3: 60.0 IQR: 29.0
Límite inferior: -12.5 Límite superior: 103.5
Outliers IQR: 0


Z-Score

In [16]:
mu = edad.mean()
sd = edad.std(ddof=0)
z = (edad - mu) / sd
out_z = edad[z.abs() > 3]

print("Media:", mu, "Std:", sd)
print("Outliers Z-score (|z|>3):", len(out_z))

Media: 46.47282608695652 Std: 16.434961430761675
Outliers Z-score (|z|>3): 0


segun el critero aplicado a Z-Score no hay valores extremos, entonces se considera correcto

Capping

In [17]:
edad_cap = edad.clip(lower=li, upper=ls)

resumen = pd.DataFrame({
    "estadistico": ["media", "mediana", "std", "min", "max"],
    "antes": [edad.mean(), edad.median(), edad.std(ddof=0), edad.min(), edad.max()],
    "despues_winsor": [edad_cap.mean(), edad_cap.median(), edad_cap.std(ddof=0), edad_cap.min(), edad_cap.max()]
})

display(resumen.round(2))

,estadistico,antes,despues_winsor
0,media,46.47,46.47
1,mediana,48.00,48.00
2,std,16.43,16.43
3,min,18.00,18.00
4,max,75.00,75.00


como en el Z-Score no hay outliers, en la winsorización o capping los datos no cambian

In [18]:
print("Outliers IQR:", len(out_iqr))
print("Outliers Z-score:", len(out_z))
print("¿Cambió algo con capping?:", not resumen["antes"].equals(resumen["despues_winsor"]))

Outliers IQR: 0
Outliers Z-score: 0
¿Cambió algo con capping?: False
